# Model Training - Demand Forecasting
This notebook connects to your local MySQL database, extracts historical booking data, trains a Random Forest model, and saves it to disk as `model_bundle.joblib`.

In [4]:
import pandas as pd
import numpy as np
import pymysql
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib
import re

# Connect to MySQL Database
# Update password if your root user has one!
connection = pymysql.connect(
    host='127.0.0.1',
    user='root',
    password='student',
    database='afterhours'
)

query = """
SELECT booking_date, booking_time, party, mood, created_at 
FROM bookings 
WHERE status NOT IN ('cancelled', 'no_show')
"""
df_raw = pd.read_sql(query, connection)
connection.close()

print(f"Loaded {len(df_raw)} valid bookings.")

Loaded 2887 valid bookings.


C:\Users\ASUS\AppData\Local\Temp\ipykernel_13676\3716326401.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_raw = pd.read_sql(query, connection)


In [5]:
# Helper functions to parse data
def parse_hour(raw):
    if not raw: return 12
    s = str(raw).strip().lower()
    m = re.search(r'(\d{1,2})(?::(\d{2}))?\s*(am|pm)?', s)
    if not m: return 12
    h = int(m.group(1))
    ampm = m.group(3)
    if ampm == "pm" and h < 12: h += 12
    if ampm == "am" and h == 12: h = 0
    return h

# Process dates and hours
df_raw['date'] = pd.to_datetime(df_raw['booking_date']).dt.date
df_raw['date'] = df_raw['date'].fillna(pd.to_datetime(df_raw['created_at']).dt.date)
df_raw['hour'] = df_raw['booking_time'].apply(parse_hour)
df_raw['party'] = df_raw['party'].fillna(2).astype(int)

df_raw['dow'] = pd.to_datetime(df_raw['date']).dt.weekday
df_raw['dow_js'] = (df_raw['dow'] + 1) % 7 # Map to JS days where Sunday=0

In [6]:
# Aggregate hourly demand
hourly_demand = df_raw.groupby(['date', 'dow_js', 'hour']).size().reset_index(name='bookings')

X = hourly_demand[['dow_js', 'hour']]
y = hourly_demand['bookings']

print("Training model...")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

preds = model.predict(X)
mae = mean_absolute_error(y, preds)
print(f"Model trained successfully! MAE: {mae:.2f}")

Training model...
Model trained successfully! MAE: 1.04


In [11]:
from sklearn.metrics import r2_score

r2_score(y,preds)

0.8841284775760091

In [7]:
# Calculate global averages to bundle with the model
total_bookings = len(df_raw) or 1

# Party proportions
p_counts = {1:0, 2:0, 3:0, 5:0}
for p in df_raw['party']:
    if p == 1: p_counts[1] += 1
    elif p == 2: p_counts[2] += 1
    elif p in (3, 4): p_counts[3] += 1
    else: p_counts[5] += 1
        
party_mix_prop = {
    "p1": p_counts[1] / total_bookings,
    "p2": p_counts[2] / total_bookings,
    "p34": p_counts[3] / total_bookings,
    "p5": p_counts[5] / total_bookings,
}

# Mood proportions
KNOWN_MOODS = ["Date", "Studying", "Hanging Out", "Creative"]
valid_moods = df_raw['mood'].dropna()
mood_counts = {m: 0 for m in KNOWN_MOODS}
for m in valid_moods:
    if m in mood_counts: mood_counts[m] += 1

total_m = sum(mood_counts.values()) or 1
mood_mix_prop = {k: v / total_m for k, v in mood_counts.items()}

daily_avg = df_raw.groupby('date').size().mean() if not df_raw.empty else 1
avg_party = df_raw['party'].mean() if not df_raw.empty else 2.0

In [8]:
# Bundle everything together and save to disk
bundle = {
    "model": model,
    "mae": mae,
    "sample_size": len(hourly_demand),
    "party_mix_prop": party_mix_prop,
    "mood_mix_prop": mood_mix_prop,
    "daily_avg": daily_avg,
    "avg_party": avg_party
}

joblib.dump(bundle, 'model_bundle.joblib')
print("Model bundle saved to 'model_bundle.joblib'!")

Model bundle saved to 'model_bundle.joblib'!
